# Mastering Retrieval-Augmented Generation (RAG): An Educational Guide

Welcome to this comprehensive tutorial on **Retrieval-Augmented Generation (RAG)**. This notebook is designed to teach you RAG concepts step-by-step, explaining the *why*, the *what*, and the *actual reason* behind each phase of a RAG pipeline.

### What is RAG? (Retrieval-Augmented Generation kya hai?)
RAG is a technique where we take a user's question, **retrieve** relevant documents or information from an external source (like PDFs, databases, websites), **augment** (combine) the question with this retrieved context, and then pass it to a Large Language Model (**LLM**) to **generate** a factually accurate answer.

### Why do we need RAG? (RAG ki zaroorat kyu hai?)
1. **Hallucination Control**: LLMs are creative and prone to making up facts (hallucinating) when they don't know the answer. By providing exact context, we restrict the LLM to only answer based on the provided documents.
2. **Private/Custom Data**: LLMs are trained on public internet data. They do not know about your private company documents, latest news, or personal files. RAG allows LLMs to access this custom data without retraining or fine-tuning.
3. **Cost and Speed**: Fine-tuning an LLM on new data is expensive, slow, and hard to update. With RAG, updating the model's knowledge is as simple as adding or replacing files in your database.

## Phase 1: Data Ingestion (Loading Documents)

Before we can retrieve information, we must load it. In this step, we read files (PDFs) from our disk and convert them into raw text documents.

### Why are we doing this?
We need to parse PDFs and convert them into a structured format (usually LangChain `Document` objects) that contain the page content and metadata (like page numbers, file names).

### What happens if we don't do this?
If we don't parse the files, we cannot extract text, which means we can't create embeddings or search through them. We'd have raw binary PDF files which LLMs cannot understand.

In [5]:
import os
from pathlib import Path
# PyPDFLoader allows us to load PDF page-by-page and extracts page number metadata automatically.
from langchain_community.document_loaders import PyPDFLoader
# RecursiveCharacterTextSplitter splits long documents into smaller chunks based on natural delimiters.
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("Core ingestion libraries imported successfully!")


C:\Users\Abhishek\AppData\Local\Temp\ipykernel_18984\3887581183.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
e:\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Core ingestion libraries imported successfully!


In [6]:
def process_all_pdfs(pdf_directory):
    """
    Scans the directory for PDF files, loads their text content page-by-page,
    and attaches custom metadata (file name and file type) to each document.
    """
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Recursively find all PDF files in the target directory
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process.")
    
    for pdf_file in pdf_files:
        print(f"Processing: {pdf_file.name}")
        try:
            # Initialize the PDF loader for the file path
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # We add custom metadata attributes: source_file and file_type
            # WHY: This helps us cite the source of the answer later (e.g., "Answer found in file.pdf, Page 4").
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Successfully loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error loading {pdf_file.name}: {e}")
    
    print(f"\nTotal pages loaded across all files: {len(all_documents)}")
    return all_documents

# Run document loading. 
# Note: We store PDFs in a '../data' folder relative to this notebook
all_pdf_documents = process_all_pdfs("../data")


Found 2 PDF files to process.
Processing: 1706.03762v7.pdf
  ✓ Successfully loaded 15 pages
Processing: Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks — Original RAG paper..pdf
  ✓ Successfully loaded 19 pages

Total pages loaded across all files: 34


## Phase 2: Document Chunking (Text Splitting)

Now we have raw pages of text. But standard book pages can be too long or cover multiple different topics. We need to split them into smaller, logically coherent pieces called **chunks**.

### Why are we doing this? (Chunking kyu karte hain?)
1. **Context Limits**: LLMs have a limit on how much text they can read at once (Context Window). We cannot pass an entire book into the prompt.
2. **Precision**: If a PDF has 100 pages, and the answer to our question is on Page 45, we only want to retrieve Page 45's specific paragraph, not the whole document. This keeps context relevant and noise-free.

### Key Parameters of Recursive Text Splitter:
- **`chunk_size`**: The maximum number of characters in a single chunk (e.g., 1000 characters).
  - *What if it's too large?* The chunk contains multiple topics, making retrieval less precise.
  - *What if it's too small?* The chunk splits a sentence in half, causing the model to lose the core context.
- **`chunk_overlap`**: The number of characters shared between consecutive chunks (e.g., 200 characters).
  - *Why do we need overlap?* Overlap guarantees that if a vital sentence spans the boundary between Chunk A and Chunk B, it is fully preserved in at least one of the chunks, preventing context splitting.

In [7]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """
    Splits raw document pages into smaller semantic chunks.
    Using RecursiveCharacterTextSplitter ensures we split by double newlines,
    single newlines, spaces, and finally characters to keep paragraphs together.
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]  # Order of priority for splitting
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} pages into {len(split_docs)} chunks.")
    
    # Show an example of a chunk to see how it looks
    if split_docs:
        print(f"\nExample Chunk:")
        print(f"--- Content (first 200 chars) ---\n{split_docs[0].page_content[:200]}...")
        print(f"--- Metadata ---\n{split_docs[0].metadata}")
    
    return split_docs

# Perform splitting on loaded documents
chunks = split_documents(all_pdf_documents)


Split 34 pages into 144 chunks.

Example Chunk:
--- Content (first 200 chars) ---
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
...
--- Metadata ---
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\1706.03762v7.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': '1706.03762v7.pdf', 'file_type': 'pdf'}


## Phase 3: Text Embeddings (Mathematical Representation)

Computers do not understand English text, but they excel at numbers. We need to convert each text chunk into a list of numbers called a **Vector Embedding**.

### What is an Embedding? (Embedding kya hota hai?)
An embedding is a numerical representation of semantic meaning. In this vector space, texts with similar meanings are close to each other. 
For example, "king" and "queen" are close to each other, while "king" and "banana" are far apart.

### Why do we embed? (Embeddings kyu banate hain?)
Keyword matching (like Ctrl+F) fails when synonyms are used. For example, if a user queries "AI assistance" but the document contains "intelligent agent", keyword search will miss it. Embeddings capture the underlying **meaning**, enabling **semantic search**.

In [8]:
import numpy as np
# SentenceTransformer is a framework to load state-of-the-art embedding models.
from sentence_transformers import SentenceTransformer
# chromadb is our vector database which stores vectors and performs fast distance calculations.
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

print("Embedding and Database libraries imported successfully!")


Embedding and Database libraries imported successfully!


In [9]:
class EmbeddingManager:
    """
    Manages loading the embedding model and generating dense vectors for texts.
    We use 'all-MiniLM-L6-v2', a fast, lightweight sentence transformer model.
    """
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            # get_embedding_dimension returns the size of the vector list (384 values for MiniLM)
            print(f"Model loaded successfully. Dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model is not loaded!")
        
        # Generate vectors (embeddings) for all texts
        embeddings = self.model.encode(texts, show_progress_bar=True)
        return embeddings

# Initialize embedding manager
embedding_manager = EmbeddingManager()


Loading embedding model: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2920.31it/s]


Model loaded successfully. Dimension: 384


## Phase 4: Vector Databases (Storage & Indexing)

Once we have vector embeddings for our chunks, we need a special database to store them. This database is optimized for search by vectors, not keywords.

### Why do we need a Vector DB?
A standard SQL database scans every row to check constraints, which is slow for vectors. Vector databases use algorithms like **HNSW (Hierarchical Navigable Small World)** to index vectors, enabling sub-millisecond retrieval of the most similar items across millions of entries.

In [10]:
class VectorStore:
    """
    Interfaces with ChromaDB to persist document chunks and their embeddings.
    """
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            # Create folder if it doesn't exist
            os.makedirs(self.persist_directory, exist_ok=True)
            # Initialize ChromaDB client to write data permanently on disk
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Fetch or create a collection (equivalent to a table in SQL)
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG tutorial"}
            )
            print(f"Vector DB Collection '{self.collection_name}' initialized.")
            print(f"Current document count in DB: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("Documents count must match embeddings count!")
            
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        # Format documents and metadata into flat lists required by ChromaDB API
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID for each chunk
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Metadata dictionaries can only hold string, int, float, or bool values
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())
            
        # Insert chunks and vectors into database
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} chunks to Vector DB.")
            print(f"Updated document count: {self.collection.count()}")
        except Exception as e:
            print(f"Error saving documents to vector store: {e}")
            raise

# Create instance of vector database client
vectorstore = VectorStore()


Vector DB Collection 'pdf_documents' initialized.
Current document count in DB: 288


### Inserting Chunks into Database

We will now run the ingestion pipeline: convert the split text chunks into vectors using `embedding_manager` and save them into our `vectorstore` database.

In [11]:
# Extract text content from document chunks
chunk_texts = [doc.page_content for doc in chunks]

# Generate embeddings for chunks
chunk_embeddings = embedding_manager.generate_embeddings(chunk_texts)

# Add chunks to ChromaDB vector store
vectorstore.add_documents(chunks, chunk_embeddings)


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches: 100%|██████████| 5/5 [00:05<00:00,  1.09s/it]


Successfully added 144 chunks to Vector DB.
Updated document count: 432


## Phase 5: Query Retrieval (Semantic Search)

This is the **Retrieval** part of RAG. When a user asks a question, we convert the question into a vector and compare it against database vectors using distance metrics.

### How are vectors compared?
ChromaDB calculates the distance between vectors. Standard metrics include:
- **L2 Distance (Euclidean)**: Direct straight-line distance.
- **Cosine Distance**: Measures direction of vectors (independent of text length). 
  - *Similarity Score* is calculated as `1 - Cosine Distance`. A higher score represents a closer semantic match.

In [12]:
class RAGRetriever:
    """
    Handles converting a user query to embeddings and querying ChromaDB to find similar chunks.
    """
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        print(f"Retrieving context for query: '{query}'")
        
        # 1. Convert user query to vector
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # 2. Search collection for top_k matches
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            retrieved_docs = []
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                # 3. Format results and filter using similarity score
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                print(f"Retrieved {len(retrieved_docs)} chunks from database.")
            else:
                print("No matching documents found in database.")
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

# Initialize retriever
rag_retriever = RAGRetriever(vectorstore, embedding_manager)


### Testing Semantic Retrieval

Let's verify that retrieval works correctly by querying terms from our loaded PDFs (Attention mechanism and original RAG papers).

In [13]:
# Querying details about the 'Attention Is All You Need' paper
results_1 = rag_retriever.retrieve("What is attention mechanism?", top_k=2)
for doc in results_1:
    print(f"\nRank {doc['rank']} (Score: {doc['similarity_score']:.4f}) in {doc['metadata']['source_file']}:")
    print(doc['content'][:250] + "...")


Retrieving context for query: 'What is attention mechanism?'


Batches: 100%|██████████| 1/1 [00:00<00:00, 36.71it/s]

Retrieved 2 chunks from database.

Rank 1 (Score: 0.2714) in 1706.03762v7.pdf:
3.2 Attention
An attention function can be described as mapping a query and a set of key-value pairs to an output,
where the query, keys, values, and output are all vectors. The output is computed as a weighted sum
3...

Rank 2 (Score: 0.2714) in 1706.03762v7.pdf:
3.2 Attention
An attention function can be described as mapping a query and a set of key-value pairs to an output,
where the query, keys, values, and output are all vectors. The output is computed as a weighted sum
3...


In [14]:
# Querying the definition of RAG from the original RAG paper
results_2 = rag_retriever.retrieve("Retrieve definition of RAG sequence model",top_k=2)

for doc in results_2:
    print(f"\nRank {doc['rank']} (Score: {doc['similarity_score']:.4f}) in {doc['metadata']['source_file']}:")
    print(doc['content'][:250] + "...")


Retrieving context for query: 'Retrieve definition of RAG sequence model'


Batches: 100%|██████████| 1/1 [00:00<00:00, 16.44it/s]

Retrieved 2 chunks from database.

Rank 1 (Score: 0.2689) in Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks — Original RAG paper..pdf:
2 Methods
We explore RAG models, which use the input sequencex to retrieve text documentsz and use them
as additional context when generating the target sequence y. As shown in Figure 1, our models
leverage two components: (i) a retriever pη(z|x) wit...

Rank 2 (Score: 0.2689) in Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks — Original RAG paper..pdf:
2 Methods
We explore RAG models, which use the input sequencex to retrieve text documentsz and use them
as additional context when generating the target sequence y. As shown in Figure 1, our models
leverage two components: (i) a retriever pη(z|x) wit...


## Phase 6: Generation (LLM Integration)

This is the final phase of RAG. We take the user's question, retrieve the top matches, format them into a prompt template, and send them to the Large Language Model.

### Prompt Template Structure:
```
Answer the Question based only on the provided Context:
Context: [Retrieved chunk 1 + Retrieved chunk 2]
Question: [User question]
Answer:
```

### Model Updates:
- We use **`llama-3.1-8b-instant`** (hosted on Groq) because `gemma2-9b-it` has been decommissioned by the API provider.
- We use **`langchain_groq.ChatGroq`** for API calls.
- We import prompts and message schemas safely from **`langchain_core`** to prevent deprecation import warnings.

In [15]:
import os
from dotenv import load_dotenv

# Load environment variables from the .env file (containing GROQ_API_KEY)
load_dotenv()

from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

print("LLM environment and modules loaded!")


LLM environment and modules loaded!


In [16]:
class GroqLLM:
    """
    Manages ChatGroq integration, providing methods to generate responses
    using structured instruction templates.
    """
    def __init__(self, model_name: str = "llama-3.1-8b-instant", api_key: str = None):
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key missing. Set GROQ_API_KEY in .env or pass it to __init__.")
            
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,  # Low temperature ensures more factual responses, less creativity
            max_tokens=1024
        )
        print(f"Initialized ChatGroq client with model: {self.model_name}")

    def generate_response(self, query: str, context: str) -> str:
        """
        Formats the instruction prompt incorporating context and query, then runs generation.
        """
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following retrieved context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear explanation based on context. If the context doesn't contain enough information to answer, state that explicitly."""
        )
        
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error generating response: {e}"

# Initialize LLM client wrapper
groq_llm = GroqLLM()


Initialized ChatGroq client with model: llama-3.1-8b-instant


## Phase 7: Building Pipelines (Simple vs. Advanced RAG)

Let's build functions that orchestrate the full workflow: **Query -> Retrieve Chunks -> Format Prompt -> Run LLM -> Output Answer**.

In [17]:
def rag_simple(query, retriever, llm_wrapper, top_k=3):
    """
    Basic RAG flow.
    """
    # 1. Retrieve raw text chunks
    results = retriever.retrieve(query, top_k=top_k)
    
    # 2. Concatenate chunk contents
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found in database to answer this question."
        
    # 3. Call LLM with augmented prompt
    return llm_wrapper.generate_response(query, context)

# Run simple RAG test
answer = rag_simple("What is attention mechanism?", rag_retriever, groq_llm)
print("\n--- RAG Answer ---\n", answer)


Retrieving context for query: 'What is attention mechanism?'


Batches: 100%|██████████| 1/1 [00:00<00:00, 32.08it/s]

Retrieved 3 chunks from database.



--- RAG Answer ---
 Based on the provided context, an attention mechanism can be described as a function that maps a query and a set of key-value pairs to an output. The output is computed as a weighted sum, where the query, keys, values, and output are all vectors. This suggests that the attention mechanism is used to selectively focus on certain parts of the input (key-value pairs) based on the query, and produce a weighted output.


### Creating an Advanced RAG Pipeline

Production-grade RAG systems require more than just basic answers. An **Advanced RAG Pipeline** should provide:
1. **Citations (Sources)**: Showing exactly which files and page numbers contributed to the answer.
2. **Confidence Scores**: Measuring how confident the retriever was (similarity scores).
3. **Response Summarization**: Providing quick executive summaries of long answers.
4. **Conversational Memory**: Storing session query history.

In [18]:
import time

class AdvancedRAGPipeline:
    """
    An advanced pipeline showing citation tracking, confidence thresholds,
    conversational logs, and answer summaries.
    """
    def __init__(self, retriever: RAGRetriever, llm_wrapper: GroqLLM):
        self.retriever = retriever
        self.llm_wrapper = llm_wrapper
        self.history = []

    def query(self, question: str, top_k: int = 5, min_score: float = 0.1, summarize: bool = False) -> Dict[str, Any]:
        # 1. Retrieve chunks with a minimum similarity threshold
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        
        if not results:
            return {
                'answer': 'No relevant sources met the confidence threshold.',
                'sources': [],
                'confidence': 0.0,
                'summary': None
            }
            
        # 2. Build context and compile citations
        context = "\n\n".join([doc['content'] for doc in results])
        sources = [{
            'source': doc['metadata'].get('source_file', 'unknown'),
            'page': doc['metadata'].get('page', 'unknown'),
            'similarity_score': doc['similarity_score']
        } for doc in results]
        
        # Confidence score is taken as the maximum similarity score retrieved
        max_confidence = max([doc['similarity_score'] for doc in results])
        
        # 3. Call LLM wrapper to generate grounded answer
        answer = self.llm_wrapper.generate_response(question, context)
        
        # 4. Format Citations
        citations = [f"- Source: {src['source']} (Page {src['page'] + 1}) | Score: {src['similarity_score']:.4f}" 
                     for src in sources]
        answer_with_citations = answer + "\n\n**Citations:**\n" + "\n".join(citations)
        
        # 5. Optional summarization
        summary = None
        if summarize:
            summary_prompt = f"Summarize the key points of this answer in 2 bullet points:\n{answer}"
            # We can use our underlying Groq client for direct prompts
            summary_resp = self.llm_wrapper.llm.invoke([HumanMessage(content=summary_prompt)])
            summary = summary_resp.content
            
        # 6. Append to conversational logs
        session_log = {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'confidence': max_confidence,
            'summary': summary
        }
        self.history.append(session_log)
        
        return session_log

# Instantiate the advanced pipeline
advanced_pipeline = AdvancedRAGPipeline(rag_retriever, groq_llm)


In [19]:
# Run query against advanced pipeline
result = advanced_pipeline.query(
    question="What is RAG sequence model , also give its formula",
    top_k=3,
    min_score=0.05,
    summarize=True
)

print("\n--- Advanced RAG Output ---")
print("\n[Answer]:\n", result['answer'])
print("\n[Confidence Score]:", result['confidence'])
print("\n[Executive Summary]:\n", result['summary'])


Retrieving context for query: 'What is RAG sequence model , also give its formula'


Batches: 100%|██████████| 1/1 [00:00<00:00, 32.84it/s]

Retrieved 3 chunks from database.



--- Advanced RAG Output ---

[Answer]:
 Based on the provided context, it appears that RAG stands for "Retrieval-Augmented Generator" or "Reformer-Augmented Generator" but more likely "Retrieval-Augmented Generator" as it is a sequence model that combines a generator with a retriever. 

The generator is a sequence model that generates text based on the input, while the retriever is a model that retrieves relevant information from a knowledge base to aid the generator. The RAG model uses this retrieved information to improve the generated text.

Unfortunately, the provided context does not contain a specific formula for the RAG sequence model. However, the general architecture of RAG can be described as follows:

1. Input: The input to the RAG model is a query or prompt.
2. Retrieval: The retriever model retrieves relevant information from a knowledge base based on the input query.
3. Generator: The generator model generates text based on the input query and the retrieved information.
